# 🐻‍❄️ 투빅스 정규세션 4주차 1교시 Image Generative AI 코드 과제 🐻‍❄️



## 과제 목적
이번 과제에서는 이미지 생성 모델의 세 가지 큰 축인 **VAE**, **GAN**, **Diffusion**의 핵심 구현을 직접 코드로 뜯어보고, 마지막으로 Hugging Face의 사전학습 Diffusion 모델을 API처럼 불러와 실습해봅니다.
끝에는 직접 자기가 원하는 모델을 선택 후 써보는 경험까지 고안해보며 활용 능력까지 얻을 수 있도록 해봅니다.




## 제출물

1. 빈칸을 모두 채우고 **전부 실행한** 이 노트북 (출력 포함)
2. 기술블로그 '과제 복습' 섹션 — 마지막에 모델 활용한 [hugging face] question 4 부분만




- **VAE (Variational AutoEncoder)**: 입력 데이터를 확률적인 잠재 공간(latent space)으로 압축한 뒤, 그 분포에서 샘플링하여 새로운 데이터를 생성하는 모델
- **GAN (Generative Adversarial Network)**: Generator(위조지폐범)와 Discriminator(경찰)가 서로 경쟁하며 학습하는 모델
- **Diffusion Model**: 이미지에 점진적으로 노이즈를 주입(Forward process)한 뒤, 그 노이즈를 단계적으로 제거(Reverse process)하도록 학습하여 무작위 노이즈로부터 이미지를 생성하는 모델

Part 1~3에서는 세 모델을 MNIST로 직접 구현/학습하며 원리를 비교하고, Part 4에서는 Hugging Face `diffusers` 라이브러리로 실제 학습된 대형 Diffusion 모델을 실습합니다.


### **모듈 임포트 및 디바이스 설정**

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torchvision.utils import make_grid
import matplotlib.pyplot as plt
import torch.nn.functional as F
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)


### **데이터셋 준비 (MNIST dataset)**

In [ ]:
batch_size = 128
epochs = 15

transform = transforms.Compose([
    transforms.ToTensor()
])

dataset = torchvision.datasets.MNIST(
    root='./data',
    train=True,
    download=True,
    transform=transform
)

# train / val split
train_size = int(0.9 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = torch.utils.data.random_split(
    dataset,
    [train_size, val_size]
)

train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

val_loader = torch.utils.data.DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False
)


## **Part 1. VAE (Variational AutoEncoder) 모델 정의 및 학습**

* Encoder: 입력 데이터 x를 받아 잠재 공간(Latent Space)의 파라미터를 추출
  * 고차원의 입력데이터를 저차원의 평균과 로그 분산으로 변환
    * 💡 빈칸 채우기 hint) `self.fc1`을 거쳐 특징을 추출한 뒤, 두 개의 서로 다른 선형 레이어를 통해 각각 평균과 로그 분산을 계산

* Latent Space & Reparameterization
  * 잠재 변수 z를 직접 고정된 값으로 뽑는 것이 아니라, encoder가 만든 분포(평균, 로그 분산)에서 샘플링함
  * Reparameterization Trick: backpropagation는 샘플링 연산을 통과할 수 없다는 문제를 해결하기 위해 샘플링 과정을 미분 가능한 형태로 만들기 위한 수식

* Decoder
  * 샘플링된 잠재 변수 z를 다시 원래의 입력 데이터 차원으로 복원


참고 코드: https://github.com/NoviceStone/VAE/blob/master/models.py

In [ ]:
class VAE(nn.Module):

    def __init__(self, input_size=784, hidden_size=400, latent_size=20):
        super(VAE, self).__init__()

        # Encoder: layers
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.fc21 = nn.Linear(hidden_size, latent_size)  # 평균(mu)
        self.fc22 = nn.Linear(hidden_size, latent_size)  # 로그 분산(logvar)

        # Decoder: layers
        self.fc3 = nn.Linear(latent_size, hidden_size)
        self.fc4 = nn.Linear(hidden_size, input_size)

    def encode(self, x):
        h1 = F.relu(self.fc1(x))
        mu = self.fc21(h1)        # 🤔 빈칸을 채워주세요 (h1을 받아 평균을 계산하는 레이어)
        logvar = self.fc22(h1)    # 🤔 빈칸을 채워주세요 (h1을 받아 로그 분산을 계산하는 레이어)
        return mu, logvar

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std     # 🤔 빈칸을 채워주세요 (Reparameterization Trick 수식)

    def decode(self, z):
        h3 = F.relu(self.fc3(z))
        return torch.sigmoid(self.fc4(h3))

    def forward(self, x):
        mu, logvar = self.encode(x.view(-1, 784))
        z = self.reparameterize(mu, logvar)
        return self.decode(z), mu, logvar


**✍️ 위에서 채운 `encode` / `reparameterize` 코드가 각각 어떤 기능을 하고, 코드적으로 어떻게 구성되어 있는지 간단히 설명해주세요.**

`encode`는 이미지 한 장을 잠재 분포 q(z|x)의 파라미터로 바꿔주는 부분이다. 784차원 입력이 `fc1` + ReLU를 지나 400차원 특징 `h1`이 되고, 여기서 갈래가 둘로 나뉜다. `fc21`은 평균 mu(20차원), `fc22`는 로그 분산 logvar(20차원)를 뽑는다. 두 레이어가 같은 `h1`을 공유하되 가중치는 서로 다르기 때문에, 하나의 특징에서 "중심이 어디냐"와 "얼마나 퍼져 있냐"를 따로 배우게 된다.

분산을 그대로 출력하지 않고 굳이 로그 분산으로 두는 이유는, 분산은 항상 양수여야 하는데 선형 레이어의 출력에는 그런 제약을 걸 수 없기 때문이다. 로그 스케일로 받아두면 나중에 `exp`를 씌우는 순간 양수가 보장되고, 값의 범위도 훨씬 안정적으로 다뤄진다.

`reparameterize`는 그 분포에서 z를 실제로 하나 뽑는 부분인데, `torch.normal(mu, std)`처럼 직접 샘플링해버리면 그 지점에서 계산 그래프가 끊겨 encoder까지 gradient가 흐르지 못한다. 그래서 무작위성만 `eps ~ N(0, I)`로 따로 떼어내고, `std = exp(0.5 * logvar)`를 곱한 뒤 `mu`를 더하는 식(`mu + eps * std`)으로 z를 만든다. 이렇게 해도 z의 분포는 여전히 N(mu, std²)로 같지만, mu와 std 입장에서는 그냥 곱셈과 덧셈이라 미분이 그대로 통과한다.


### **VAE loss (ELBO)**
* Reconstruction Loss: 복원된 이미지(`recon_x`)가 원본 이미지(`x`)와 얼마나 다른지 측정
* KL Divergence: 잠재공간의 분포가 표준 정규분포와 얼마나 다른지 측정


In [ ]:
def elbo_loss(x, recon_x, mu, logvar):

    B = x.size(0)

    x = x.view(-1, 784)

    BCE = F.binary_cross_entropy(recon_x, x, reduction='sum') / B  # 🤔 빈칸을 채워주세요

    KLD = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / B  # 🤔 빈칸을 채워주세요

    total_loss = BCE + KLD  # 🤔 빈칸을 채워주세요

    return total_loss


**✍️ 위에서 채운 `elbo_loss` 코드가 어떤 기능을 하고, 코드적으로 어떻게 구성되어 있는지 간단히 설명해주세요.**

ELBO를 최대화하는 대신 부호를 뒤집어 최소화할 손실로 만든 함수이고, 두 항으로 되어 있다.

`BCE`는 복원 항이다. MNIST 픽셀값이 0~1 사이라 각 픽셀을 베르누이 확률로 보고, decoder가 sigmoid로 뱉은 `recon_x`를 원본 `x`와 픽셀 단위로 비교한다. 인자 순서는 (예측, 정답)이라 `recon_x`가 앞에 온다. `reduction='sum'`으로 784개 픽셀을 전부 더한 뒤 배치 크기 `B`로만 나누는데, 이러면 "이미지 한 장당 로그 가능도"라는 원래 의미가 유지된다.

`KLD`는 정규화 항이다. q(z|x) = N(mu, sigma²)와 사전분포 p(z) = N(0, I) 사이의 KL divergence인데, 양쪽이 다 가우시안이라 닫힌 형태로 정리되어 `-0.5 * sum(1 + logvar - mu² - exp(logvar))`가 된다. mu가 0에서 멀어지거나 분산이 1에서 벗어날수록 커지는 구조라, 잠재 분포를 표준정규분포 쪽으로 끌어당기는 역할을 한다. 이쪽도 `B`로 나눠 복원 항과 스케일을 맞춰준다.

둘을 더한 값이 -ELBO이고, 이걸 줄이는 것이 곧 ELBO를 키우는 것이다.


## **🙋 [VAE] Question 1**
VAE의 ELBO Loss에 KL Divergence 항이 포함되는 이유를 설명해주세요.

우선 유도 과정상 처음부터 붙어 있는 항이다. log p(x)를 직접 계산할 수 없어서 Jensen 부등식으로 하한을 잡으면 log p(x) ≥ E_q[log p(x|z)] - KL(q(z|x) || p(z))가 나오는데, 앞이 복원 항이고 뒤가 KL 항이다. 정확히는 log p(x) = ELBO + KL(q(z|x) || p(z|x))라서, KL 항을 줄이는 것은 근사 분포 q를 실제 사후분포에 가깝게 맞추는 작업이기도 하다.

의미를 놓고 보면 이유가 더 분명하다. 복원 항만 두고 학습하면 encoder는 손실을 줄이기 위해 각 샘플을 잠재 공간의 아주 좁은 한 점으로 몰아넣고 분산을 0에 가깝게 만들어버린다. 그러면 모델은 사실상 일반 AutoEncoder가 되고, 잠재 공간은 학습 데이터가 찍힌 자리만 띄엄띄엄 채워진 채 사이사이가 비어버린다. 이 상태에서 `torch.randn(64, 20)`으로 아무 z나 뽑아 decode하면 학습 때 한 번도 지나가지 않은 빈 구역이라 의미 없는 얼룩이 나온다.

KL 항은 모든 q(z|x)를 N(0, I) 쪽으로 끌어당겨서 이 문제를 막는다. 각 샘플의 잠재 분포가 적당한 분산을 유지한 채 겹치도록 강제하니 공간이 구멍 없이 메워지고, 그 덕분에 표준정규분포에서 z를 뽑아 넣는 샘플링이 성립한다. 즉 KL 항은 "복원을 방해하는 페널티"가 아니라 VAE를 생성 모델로 만들어주는 조건에 가깝다. 다만 너무 세면 반대로 latent를 아예 무시해버리는 posterior collapse가 생겨서, beta-VAE처럼 가중치를 조절하는 변형들이 나온 것이다.


In [ ]:
vae = VAE().to(device)
optimizer = optim.Adam(vae.parameters(), lr=1e-3)

for epoch in range(epochs):

    # ========== Train ==========
    vae.train()
    train_loss = 0

    for x, _ in train_loader:

        x = x.to(device)

        recon, mu, logvar = vae(x)

        loss = elbo_loss(x, recon, mu, logvar)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    train_loss /= len(train_dataset)

    # ========== Validation ==========
    vae.eval()
    val_loss = 0

    with torch.no_grad():
        for x, _ in val_loader:
            x = x.to(device)
            recon, mu, logvar = vae(x)
            loss = elbo_loss(x, recon, mu, logvar)
            val_loss += loss.item()

    val_loss /= len(val_dataset)

    print(f"Epoch [{epoch+1}/{epochs}] Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")


**VAE sampling 시각화**

In [ ]:
with torch.no_grad():

    z = torch.randn(64, 20).to(device)

    samples = vae.decode(z).view(-1, 1, 28, 28)

    grid = make_grid(samples, 8)

plt.imshow(grid.permute(1, 2, 0).cpu())
plt.title("VAE Samples")
plt.axis("off")
plt.show()


## **Part 2. DCGAN (Deep Convolutional GAN) 모델 정의 및 학습**
- Weight Initialization
  - 모델의 가중치는 평균이 0, 표준편차가 0.02인 정규분포로 초기화
  - 기존 GAN의 학습 불안정성을 줄이기 위해 DCGAN 논문에서 제안한 아이디어

- Generator (생성자)
  - Transposed Convolution: 저차원의 잠재 벡터를 고해상도 이미지로 확장
  - Batch Normalization: 각 layer의 출력을 정규화하여 학습의 안정성을 높이고 gradient vanishing 문제를 해결
  - Activation: 중간 layer에는 ReLU를 통해 빠른 학습을 진행하고, 마지막 layer에는 이미지의 픽셀 값 범위([-1, 1])를 맞추기 위해 Tanh를 사용

- Discriminator (판별자)
  - Strided Convolution: pooling 대신 stride를 사용하여 특징을 추출
  - 마지막 layer에서 sigmoid를 통해 Real/Fake 확률값 출력


참고 코드: https://github.com/Natsu6767/DCGAN-PyTorch/blob/master/dcgan.py

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))  # [-1, 1]
])

dataset = torchvision.datasets.MNIST(
    root='./data',
    train=True,
    download=True,
    transform=transform
)

# train / val split
train_size = int(0.9 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = torch.utils.data.random_split(
    dataset,
    [train_size, val_size]
)

train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)


In [ ]:
# weight 초기화 함수

def weights_init(m):

    classname = m.__class__.__name__

    if classname.find("Conv") != -1:
        nn.init.normal_(m.weight.data, 0.0, 0.02)

    elif classname.find("BatchNorm") != -1:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)


In [ ]:
latent_size = 20

class Generator(nn.Module):

    def __init__(self):
        super().__init__()

        self.tconv1 = nn.ConvTranspose2d(latent_size, 256, 7, 1, 0, bias=False)
        self.bn1 = nn.BatchNorm2d(256)

        self.tconv2 = nn.ConvTranspose2d(256, 128, 4, 2, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(128)

        self.tconv3 = nn.ConvTranspose2d(128, 64, 4, 2, 1, bias=False)
        self.bn3 = nn.BatchNorm2d(64)

        self.tconv4 = nn.ConvTranspose2d(64, 1, 3, 1, 1, bias=False)

    def forward(self, z):
        x = F.relu(self.bn1(self.tconv1(z)))   # 🤔 빈칸을 채워주세요 (Generator의 입력)
        x = F.relu(self.bn2(self.tconv2(x)))
        x = F.relu(self.bn3(self.tconv3(x)))
        x = torch.tanh(self.tconv4(x))   # 🤔 빈칸을 채워주세요 (마지막 layer 통과)
        return x


**✍️ 위에서 채운 `Generator.forward` 코드가 어떤 기능을 하고, 코드적으로 어떻게 구성되어 있는지 간단히 설명해주세요.**

(B, 20, 1, 1) 모양의 잠재 벡터 z를 받아 (B, 1, 28, 28) 이미지로 키워 올리는 함수다. 첫 줄의 빈칸이 z인 이유는 여기가 네트워크의 시작점이라 `tconv1`에 들어갈 게 입력 벡터밖에 없기 때문이다.

크기 변화를 따라가 보면 이렇다.

- `tconv1`: kernel 7, stride 1, padding 0 → 1×1이 한 번에 7×7로 커진다. 채널은 20 → 256.
- `tconv2`: kernel 4, stride 2, padding 1 → 7×7이 14×14. 채널 256 → 128.
- `tconv3`: 같은 설정으로 14×14 → 28×28. 채널 128 → 64.
- `tconv4`: kernel 3, stride 1, padding 1 → 크기는 28×28 그대로 두고 채널만 64 → 1로 줄인다.

중간 세 단계는 전부 ConvTranspose → BatchNorm → ReLU 순서다. BatchNorm이 각 층 출력의 분포를 잡아줘서 학습 초반에 값이 튀는 걸 막아준다.

마지막 층만 BatchNorm 없이 곧장 `tanh`로 간다. 앞서 데이터를 `Normalize((0.5,), (0.5,))`로 [-1, 1] 범위에 맞춰놨기 때문에 생성 이미지도 같은 범위로 나와야 한다. 만약 여기서 sigmoid를 쓰면 가짜 이미지만 [0, 1]에 몰리게 되고, Discriminator가 숫자 모양을 볼 필요도 없이 값의 범위만 보고 진짜와 가짜를 구분해버려서 학습이 성립하지 않는다.


In [ ]:
class Discriminator(nn.Module):

    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(in_channels=1, out_channels=64, kernel_size=4, stride=2, padding=1, bias=False)

        self.conv2 = nn.Conv2d(64, 128, 4, 2, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(128)

        self.conv3 = nn.Conv2d(128, 256, 4, 2, 1, bias=False)
        self.bn3 = nn.BatchNorm2d(256)

        self.conv4 = nn.Conv2d(256, 1, 3, 1, 0, bias=False)

    def forward(self, x):

        x = F.leaky_relu(self.conv1(x), 0.2)
        x = F.leaky_relu(self.bn2(self.conv2(x)), 0.2)
        x = F.leaky_relu(self.bn3(self.conv3(x)), 0.2)   # 🤔 빈칸을 채워주세요 (conv3 + bn3 통과)
        x = torch.sigmoid(self.conv4(x))

        return x.view(-1, 1).squeeze(1)


**✍️ 위에서 채운 `Discriminator.forward` 코드가 어떤 기능을 하고, 코드적으로 어떻게 구성되어 있는지 간단히 설명해주세요.**

Generator와 정확히 반대로, (B, 1, 28, 28) 이미지를 받아 "진짜일 확률" 스칼라 하나로 줄이는 함수다. 빈칸은 `self.bn3(self.conv3(x))`인데, 앞의 두 줄과 같은 conv → BatchNorm → LeakyReLU 패턴을 그대로 따라가면 된다.

크기는 pooling 없이 stride 2로만 줄어든다. 28 → 14 → 7 → 3으로 내려가고(7에서는 (7 + 2 - 4)/2 + 1 = 3이라 3×3이 된다), 마지막 `conv4`가 kernel 3, padding 0이라 3×3을 1×1로 만들면서 채널도 1로 줄인다. DCGAN 논문이 pooling 대신 stride를 쓰라고 한 이유는, 다운샘플링 방식 자체도 학습되는 파라미터로 두는 편이 낫기 때문이다.

활성함수로 ReLU가 아니라 `LeakyReLU(0.2)`를 쓴 것도 논문 권고다. Discriminator에서 음수 입력의 gradient가 0이 되어버리면 그 경로를 타고 Generator까지 흘러가야 할 신호가 통째로 죽는데, 기울기를 0.2만큼 남겨두면 그런 구간이 없어진다.

마지막에 sigmoid로 0~1 확률을 만들고 `view(-1, 1).squeeze(1)`로 (B, 1, 1, 1)을 (B,)로 펴는데, 이건 `nn.BCELoss`에 넘길 라벨 텐서(`torch.ones(x.size(0))`)와 모양을 맞추기 위한 처리다.


In [ ]:
G = Generator().to(device)
D = Discriminator().to(device)

G.apply(weights_init)
D.apply(weights_init)

criterion = nn.BCELoss()

optG = optim.Adam(G.parameters(), 0.0002, betas=(0.5, 0.999))
optD = optim.Adam(D.parameters(), 0.0002, betas=(0.5, 0.999))


In [ ]:
for epoch in range(epochs):

    G.train()
    D.train()

    train_lossD = 0
    train_lossG = 0

    for x, _ in train_loader:

        x = x.to(device)

        real = torch.ones(x.size(0)).to(device)
        fake = torch.zeros(x.size(0)).to(device)

        # ========== Train Discriminator ==========

        z = torch.randn(x.size(0), latent_size, 1, 1).to(device)

        fake_img = G(z)

        loss_real = criterion(D(x), real)
        loss_fake = criterion(D(fake_img.detach()), fake)

        lossD = loss_real + loss_fake

        optD.zero_grad()
        lossD.backward()
        optD.step()

        # ========== Train Generator ==========

        loss_G = criterion(D(fake_img), real)  # 🤔 빈칸을 채워주세요 (Generator는 D가 fake_img를 무엇으로 착각하길 원할까요?)

        optG.zero_grad()
        loss_G.backward()
        optG.step()

        train_lossD += lossD.item()
        train_lossG += loss_G.item()

    train_lossD /= len(train_loader)
    train_lossG /= len(train_loader)

    print(f"Epoch [{epoch+1}/{epochs}] Loss D: {train_lossD:.4f} | Loss G: {train_lossG:.4f}")


**✍️ 위에서 채운  코드가 어떤 기능을 하고, 코드적으로 어떻게 구성되어 있는지 간단히 설명해주세요.**

빈칸에 들어갈 라벨은 `fake`가 아니라 `real`이다. Generator가 바라는 건 자기가 만든 이미지를 Discriminator가 진짜(1)라고 판정해주는 것이므로, `D(fake_img)`를 정답이 1인 것처럼 두고 BCE를 계산해 그 거리를 줄인다. 라벨을 뒤집는 것만으로 목적이 정반대인 두 네트워크가 같은 손실 함수 하나를 공유하게 되는 구조다.

이게 단순한 요령이 아니라 이유가 있는 선택인데, 원래 GAN 논문의 minimax 식대로면 Generator는 log(1 - D(G(z)))를 최소화해야 한다. 문제는 학습 초반에 Discriminator가 압도적으로 우세해서 D(G(z))가 0 근처에 붙어 있을 때인데, 이 식은 그 구간에서 기울기가 거의 평평해 Generator가 배울 게 없다. 그래서 논문도 실제로는 -log D(G(z))를 최대화하는 non-saturating 형태를 쓰라고 했고, BCE에 real 라벨을 넣는 것이 정확히 그 식이 된다. 정답에서 멀수록 기울기가 커지므로 밀리고 있을 때 오히려 더 세게 학습된다.

코드 순서에서 한 가지 더 볼 점은 `detach()`의 유무다. Discriminator를 학습할 때는 `fake_img.detach()`로 그래프를 끊어 Generator까지 gradient가 가지 않게 막지만, Generator 학습 줄에서는 일부러 떼지 않는다. Discriminator를 거쳐 Generator까지 gradient가 흘러가야 하기 때문이다.


**DCGAN Sampling 시각화**

In [ ]:
G.eval()

with torch.no_grad():

    z = torch.randn(64, latent_size, 1, 1).to(device)

    samples = G(z)

grid = make_grid(samples, 8, normalize=True)

plt.imshow(grid.permute(1, 2, 0).cpu())
plt.title("DCGAN Samples")
plt.axis("off")
plt.show()


## **🙋 [GAN] Question 1**
VAE와 DCGAN으로 MNIST 이미지를 생성한 결과 VAE는 흐릿한 이미지, DCGAN은 선명한 이미지가 생성된 것을 알 수 있습니다. 그 이유를 VAE와 GAN의 특징과 학습 방식을 포함하여 설명해주세요.

차이는 "무엇을 손실로 재느냐"에서 나온다.

VAE는 복원된 이미지를 원본과 픽셀 단위로 직접 비교한다. 그런데 하나의 z에 대응할 수 있는 그럴듯한 정답은 하나가 아니다. 같은 잠재 코드가 가리키는 "3"만 해도 획이 굵은 것, 기울어진 것, 아래 고리가 큰 것이 전부 후보다. 픽셀 거리를 최소화하는 입장에서 가장 손해가 적은 답은 그중 하나를 골라 선명하게 그리는 것이 아니라, 후보들을 평균 낸 이미지를 내놓는 것이다. 한쪽을 골랐다가 틀리면 손실이 크게 늘지만, 평균을 찍으면 어느 정답에 대해서도 중간 정도의 손실만 나기 때문이다. 이렇게 여러 모양이 겹쳐진 평균 이미지는 획의 경계가 뭉개져 흐릿하게 보인다. 여기에 KL 항 때문에 z에 항상 노이즈가 섞여 들어가고, decoder는 흔들리는 z에도 무난한 출력을 내도록 학습되므로 뭉개짐이 한 번 더 커진다.

GAN은 원본과의 픽셀 거리를 아예 재지 않는다. Discriminator가 보는 것은 "이 이미지가 진짜 데이터 분포에서 나온 것 같은가" 하나뿐이고, 흐릿한 이미지는 실제 MNIST에 존재하지 않기 때문에 아주 쉽게 가짜로 잡힌다. 평균으로 도망가는 순간 손실이 오히려 커지므로 Generator는 어느 한 모드에 확실히 붙은 또렷한 샘플을 만들 수밖에 없다. 게다가 Discriminator는 고정된 척도가 아니라 Generator가 좋아지는 만큼 같이 까다로워지는 학습되는 손실 함수라서, 사람 눈에 걸리는 어색함을 계속 새로 찾아낸다.

정리하면 VAE의 흐림은 학습이 덜 된 탓이 아니라 목적함수가 평균을 보상하기 때문에 생기는 구조적인 결과이고, GAN의 선명함은 그 평균화를 명시적으로 처벌한 대가로 얻은 것이다. 대신 GAN은 분포 전체를 덮을 이유가 없어서 특정 숫자만 반복해서 내놓는 mode collapse 위험을 안고 가고, 학습도 훨씬 불안정하다.


## **🙋 [GAN] Question 2**
GAN 학습 과정에서 Generator와 Discriminator 중 한쪽이 너무 빨리 강해지면 어떤 문제가 발생할 수 있는지, Mode collapse 또는 Vanishing gradient 개념을 활용하여 설명해주세요.

GAN은 두 네트워크가 서로에게 손실 함수 역할을 해주는 구조라, 한쪽 손실이 0에 가까워지는 건 잘 되고 있다는 신호가 아니라 상대에게 줄 정보가 사라졌다는 신호에 가깝다.

**Discriminator가 너무 빨리 강해지는 경우: vanishing gradient**

Discriminator가 진짜와 가짜를 완벽히 갈라내기 시작하면 D(G(z))가 0에 딱 붙는다. 이 구간은 sigmoid의 포화 영역이라 출력이 입력 변화에 거의 반응하지 않고, Generator로 역전파되는 기울기가 사실상 0이 된다. Generator 입장에서는 "가짜라고 판정당했다"는 사실만 알 뿐 어느 방향으로 고쳐야 덜 가짜인지를 못 배운다. 로그를 보면 Loss D는 0으로 수렴하는데 Loss G만 큰 값에서 내려오지 않는 형태로 나타난다. 앞에서 non-saturating loss(BCE + real 라벨)를 쓴 것이 이 문제의 1차 완화책이고, 그 밖에는 Discriminator 업데이트 횟수나 학습률을 낮추기, one-sided label smoothing(진짜 라벨을 1 대신 0.9), 입력에 노이즈 추가, 아예 WGAN 계열로 거리 자체를 바꾸는 방법이 쓰인다.

**Generator가 너무 빨리 강해지는 경우: mode collapse**

반대로 Generator가 Discriminator의 허점을 먼저 찾으면, 그 순간 잘 통하는 출력 몇 개만 계속 뱉는 게 최적 전략이 된다. 손실 식 어디에도 "다양하게 만들어라"는 항이 없기 때문이다. MNIST라면 z를 아무리 다르게 넣어도 1과 7만 잔뜩 나오는 식으로 나타난다. 더 나쁜 건 여기서 멈추지도 않는다는 점인데, Discriminator가 뒤늦게 그 모드를 잡아내면 Generator는 다른 모드로 통째로 옮겨가고, 이게 반복되면서 손실이 수렴하지 않고 진동한다. 완화책으로는 minibatch discrimination이나 feature matching처럼 배치 안의 다양성을 Discriminator가 보게 하는 방법, unrolled GAN처럼 상대의 다음 수까지 고려하는 방법, WGAN-GP처럼 기울기 자체를 안정화하는 방법이 있다.

그래서 GAN 학습의 목표는 어느 한쪽 손실을 0으로 만드는 게 아니라, 두 손실이 비슷한 수준에서 같이 유지되게 만드는 것이다. 학습 로그를 볼 때도 값이 작아지는지가 아니라 둘의 균형이 무너지지 않는지를 본다.


## **Part 3. Diffusion Model (DDPM) 원리 구현**

- **Forward process**: 원본 이미지 $x_0$에 $t$번에 걸쳐 점진적으로 Gaussian noise를 주입하여 $x_t$를 만드는 과정. Closed-form으로 한 번에 임의의 t 시점 노이즈 이미지를 계산할 수 있음
  $$x_t = \sqrt{\bar{\alpha}_t}\, x_0 + \sqrt{1-\bar{\alpha}_t}\, \epsilon,\quad \epsilon \sim \mathcal{N}(0, I)$$

- **Reverse process**: 노이즈 $x_t$로부터 노이즈 $\epsilon$을 예측하는 신경망(주로 U-Net)을 학습시켜, $x_T$(순수 노이즈)부터 $x_0$까지 역으로 복원

- **손실함수의 단순화**: DDPM 논문의 핵심 기여 중 하나로, 복잡한 분포 비교 대신 신경망이 예측한 노이즈와 실제 주입된 노이즈 사이의 단순 MSE Loss만으로 학습이 가능함
  $$L_{simple} = \mathbb{E}_{t, x_0, \epsilon}\left[\, \| \epsilon - \epsilon_\theta(x_t, t) \|^2 \,\right]$$

이번 파트에서는 **forward process와 손실함수, 노이즈 예측 네트워크의 핵심 구조를 직접 코드로 구현**하며 원리를 이해합니다. 다만 DDPM은 실제로는 대규모 U-Net을 오랜 시간 학습해야 제대로 된 이미지가 나오기 때문에, 짧은 실습 환경(Colab)에서 처음부터 학습시키기엔 한계가 있습니다. 그래서 우리가 구현한 것과 동일한 원리로 학습된 **Hugging Face의 사전학습 MNIST DDPM 모델**을 불러와 실제 reverse process 결과를 확인해봅니다.


In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))  # [-1, 1]
])

dataset = torchvision.datasets.MNIST(
    root='./data',
    train=True,
    download=True,
    transform=transform
)

train_loader = torch.utils.data.DataLoader(
    dataset,
    batch_size=128,
    shuffle=True
)


### **Noise Schedule 정의**
$\beta_t$를 선형으로 증가시키며, $\alpha_t = 1-\beta_t$, $\bar{\alpha}_t = \prod_{s=1}^{t}\alpha_s$ 를 미리 계산해둡니다.


In [ ]:
T = 1000  # DDPM 논문 기준 총 diffusion step 수

beta_start = 1e-4
beta_end = 0.02

betas = torch.linspace(beta_start, beta_end, T).to(device)
alphas = 1. - betas
alphas_cumprod = torch.cumprod(alphas, dim=0)

sqrt_alphas_cumprod = torch.sqrt(alphas_cumprod)
sqrt_one_minus_alphas_cumprod = torch.sqrt(1. - alphas_cumprod)


def forward_diffusion(x0, t):
    """x0(원본 이미지)에 timestep t만큼의 노이즈를 주입해 x_t를 반환"""
    noise = torch.randn_like(x0)

    sqrt_ac = sqrt_alphas_cumprod[t].view(-1, 1, 1, 1)
    sqrt_omac = sqrt_one_minus_alphas_cumprod[t].view(-1, 1, 1, 1)

    x_t = sqrt_ac * x0 + sqrt_omac * noise   # 🤔 빈칸을 채워주세요 (forward process 수식)

    return x_t, noise


**✍️ 위에서 채운 `forward_diffusion` 코드가 어떤 기능을 하고, 코드적으로 어떻게 구성되어 있는지 간단히 설명해주세요.**

원본 이미지에 t단계만큼의 노이즈를 한 번에 주입해서 x_t를 만드는 함수다.

정의대로라면 forward process는 x_0에서 x_1, x_2, ... 순으로 조금씩 노이즈를 더해가는 마르코프 체인이라 x_500을 얻으려면 500번을 돌아야 한다. 그런데 가우시안에 가우시안을 더하면 다시 가우시안이라 이 500번을 하나로 합칠 수 있고, 그 결과가 alpha_bar_t(= alpha들의 누적곱)만 알면 x_0에서 바로 x_t로 점프하는 closed form이다. `alphas_cumprod`를 미리 계산해두는 이유가 이것이고, 이 성질이 없으면 배치마다 수백 번 루프를 도느라 학습 자체가 불가능하다.

코드는 그 식 x_t = sqrt(alpha_bar_t) * x_0 + sqrt(1 - alpha_bar_t) * eps를 그대로 옮긴 것이다. `sqrt_ac`가 원본 신호에 붙는 계수, `sqrt_omac`가 노이즈에 붙는 계수이고, t가 커질수록 alpha_bar가 0으로 떨어지면서 앞 계수는 줄고 뒤 계수는 1에 가까워진다. 두 계수의 제곱 합이 1이라 전체 분산이 유지되고(variance preserving), t가 커지면 결국 표준정규분포에 수렴한다.

`.view(-1, 1, 1, 1)`은 배치 안에서 샘플마다 t가 다르기 때문에 필요한 처리다. `sqrt_alphas_cumprod[t]`는 (B,) 모양인데 이걸 (B, 1, 1, 1)로 바꿔야 (B, 1, 28, 28) 이미지와 브로드캐스팅되어 샘플별로 다른 계수가 곱해진다.

마지막으로 x_t만이 아니라 `noise`도 같이 반환하는 게 중요한데, 이 노이즈가 곧 다음 단계 손실 함수의 정답 라벨이 되기 때문이다.


**Forward process 시각화**: t가 커질수록 이미지가 점점 순수 노이즈에 가까워지는 것을 확인해봅시다.

In [ ]:
x0, _ = next(iter(train_loader))
x0 = x0[:1].to(device)

timesteps_to_show = [0, 100, 300, 500, 700, 999]
fig, axes = plt.subplots(1, len(timesteps_to_show), figsize=(15, 3))

for i, t_val in enumerate(timesteps_to_show):
    t = torch.tensor([t_val]).to(device)
    x_t, _ = forward_diffusion(x0, t)
    axes[i].imshow(x_t[0, 0].cpu().numpy(), cmap="gray")
    axes[i].set_title(f"t={t_val}")
    axes[i].axis("off")

plt.show()


### **U-Net 기반 노이즈 예측 네트워크**
DDPM은 노이즈가 섞인 이미지 $x_t$와 timestep $t$를 입력받아 주입된 노이즈 $\epsilon$을 예측하는 신경망을 사용합니다. 아래는 skip connection을 포함한 축소된 U-Net 구조입니다. Downsampling에서 저장해둔 feature map을 Upsampling 단계에서 다시 이어붙여(concat), 공간 정보 손실을 줄이는 것이 핵심입니다.


In [ ]:
class TimeEmbedding(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, t):
        half_dim = self.dim // 2
        emb = np.log(10000) / (half_dim - 1)
        emb = torch.exp(torch.arange(half_dim, device=t.device) * -emb)
        emb = t[:, None].float() * emb[None, :]
        emb = torch.cat([torch.sin(emb), torch.cos(emb)], dim=-1)
        return emb


class SimpleUNet(nn.Module):

    def __init__(self, time_dim=64):
        super().__init__()

        self.time_mlp = nn.Sequential(
            TimeEmbedding(time_dim),
            nn.Linear(time_dim, time_dim),
            nn.ReLU()
        )

        # Downsampling
        self.down1 = nn.Conv2d(1, 64, 3, padding=1)                    # 28 -> 28
        self.down2 = nn.Conv2d(64, 128, 3, stride=2, padding=1)        # 28 -> 14
        self.down3 = nn.Conv2d(128, 256, 3, stride=2, padding=1)       # 14 -> 7

        self.time_proj = nn.Linear(time_dim, 256)

        # Bottleneck
        self.bottleneck = nn.Conv2d(256, 256, 3, padding=1)            # 7 -> 7

        # Upsampling (skip connection으로 채널이 2배가 됨)
        self.up1 = nn.ConvTranspose2d(256, 128, 4, stride=2, padding=1)   # 7 -> 14
        self.up_conv1 = nn.Conv2d(128 + 128, 128, 3, padding=1)           # skip(h2) concat

        self.up2 = nn.ConvTranspose2d(128, 64, 4, stride=2, padding=1)    # 14 -> 28
        self.up_conv2 = nn.Conv2d(64 + 64, 64, 3, padding=1)              # skip(h1) concat

        self.out = nn.Conv2d(64, 1, 3, padding=1)

    def forward(self, x, t):
        t_emb = self.time_mlp(t)

        # Downsampling (skip connection용으로 각 단계 출력을 저장)
        h1 = F.relu(self.down1(x))      # (B, 64, 28, 28)
        h2 = F.relu(self.down2(h1))     # (B, 128, 14, 14)
        h3 = F.relu(self.down3(h2))     # (B, 256, 7, 7)

        # timestep embedding을 bottleneck 채널에 더해줌
        t_proj = self.time_proj(t_emb).unsqueeze(-1).unsqueeze(-1)
        h3 = h3 + t_proj   # 🤔 빈칸을 채워주세요 (h3에 timestep 정보를 더해주려면?)

        h3 = F.relu(self.bottleneck(h3))

        # Upsampling + skip connection (concat)
        u1 = F.relu(self.up1(h3))                      # (B, 128, 14, 14)
        u1 = torch.cat([u1, h2], dim=1)                 # (B, 256, 14, 14)
        u1 = F.relu(self.up_conv1(u1))                  # (B, 128, 14, 14)

        u2 = F.relu(self.up2(u1))                       # (B, 64, 28, 28)
        u2 = torch.cat([u2, h1], dim=1)                  # (B, 128, 28, 28)
        u2 = F.relu(self.up_conv2(u2))                   # (B, 64, 28, 28)

        out = self.out(u2)

        return out


**✍️ 위에서 채운 `SimpleUNet.forward`의 timestep 결합 코드가 어떤 기능을 하고, 코드적으로 어떻게 구성되어 있는지 간단히 설명해주세요.**

빈칸은 `t_proj`이고, 지금이 몇 번째 스텝인지를 feature map에 실어주는 부분이다.

경로를 따라가면 스칼라 t가 먼저 `TimeEmbedding`을 지나며 sin/cos 주기가 다른 64차원 벡터가 된다. Transformer의 positional encoding과 같은 방식인데, t를 숫자 하나로 그냥 넣으면 네트워크가 100과 900의 차이를 스케일 차이로만 받아들이는 반면 이렇게 여러 주파수로 펼쳐두면 가까운 t끼리는 비슷하고 먼 t끼리는 확실히 다른 표현이 나온다. 이어서 `Linear` + `ReLU`를 한 번 태워 학습 가능한 변환을 거치고, `time_proj`가 이를 bottleneck의 채널 수인 256에 맞춘다.

여기서 나온 텐서는 (B, 256)인데 더해야 할 `h3`는 (B, 256, 7, 7)이라 모양이 안 맞는다. `unsqueeze(-1)`을 두 번 해서 (B, 256, 1, 1)로 만들면 브로드캐스팅으로 7×7 위치 전체에 같은 값이 더해진다. 결과적으로 채널마다 상수 하나씩을 얹는 셈이고, 그 상수가 t에 따라 달라지므로 뒤쪽 층들이 "지금 어느 정도로 지저분한 단계인지"를 조건으로 쓸 수 있게 된다.

이게 왜 필요하냐면, 이 네트워크는 파라미터 한 벌로 1000개의 t를 전부 담당하기 때문이다. 눈으로 보기에 비슷하게 지저분한 이미지라도 t=100이면 살짝만 걷어내야 하고 t=900이면 거의 전부가 노이즈다. t를 알려주지 않으면 모델은 모든 단계에 대한 평균적인 답만 내놓게 되고, reverse process가 제대로 굴러가지 않는다.


### **DDPM Loss**
예측한 노이즈와 실제 주입한 노이즈 사이의 MSE Loss입니다.


In [ ]:
def ddpm_loss(model, x0):
    B = x0.size(0)

    t = torch.randint(0, T, (B,), device=device).long()

    x_t, noise = forward_diffusion(x0, t)

    predicted_noise = model(x_t, t)

    loss = F.mse_loss(predicted_noise, noise)   # 🤔 빈칸을 채워주세요 (실제 노이즈 vs 예측 노이즈)

    return loss


**✍️ 위에서 채운 `ddpm_loss` 코드가 어떤 기능을 하고, 코드적으로 어떻게 구성되어 있는지 간단히 설명해주세요.**

DDPM 학습 한 스텝을 그대로 옮긴 함수이고, 세 줄이 전부다.

먼저 `torch.randint(0, T, (B,))`로 배치의 샘플마다 t를 0~999 중에서 균등하게 하나씩 뽑는다. 원래 목적함수는 모든 t에 대한 기대값이지만 매번 1000개를 다 도는 건 불가능하므로, 무작위로 하나만 골라 몬테카를로로 근사하는 것이다.

그다음 `forward_diffusion`으로 x_t와 그때 실제로 넣은 `noise`를 받는다. 정답 라벨을 사람이 붙이는 게 아니라 우리가 방금 만들어 넣은 노이즈가 그대로 정답이 된다는 게 포인트다.

마지막이 빈칸인데, 모델이 (x_t, t)만 보고 예측한 `predicted_noise`와 실제 `noise` 사이의 MSE다. `F.mse_loss`는 차이를 제곱하므로 두 인자를 바꿔 넣어도 값은 같지만, 관례대로 (예측, 정답) 순서로 적었다.

원래 variational bound에는 t마다 다른 가중치가 붙어 있는데, DDPM 논문은 그 가중치를 전부 1로 두고 이 단순한 MSE만 남긴 L_simple을 제안했다. 이론적으로는 느슨해진 형태인데도 샘플 품질이 오히려 좋아졌고, 덕분에 확산 모델 학습이 GAN 같은 균형 맞추기 없이 평범한 회귀 학습처럼 안정적으로 돌아가게 됐다.


## **🙋 [Diffusion] Question 1**
DDPM은 왜 원본 이미지 $x_0$를 직접 예측하지 않고, 이미지에 주입된 노이즈 $\epsilon$을 예측하도록 학습할까요?

수학적으로는 둘이 같은 정보를 담고 있다. x_t와 t를 알고 있으면 forward 식을 뒤집어 x_0 = (x_t - sqrt(1 - alpha_bar_t) * eps) / sqrt(alpha_bar_t)로 서로 변환되므로, 노이즈를 맞히는 것과 원본을 맞히는 것은 원리상 같은 문제다. 그런데도 노이즈 예측을 고른 데는 이유가 있다.

첫째, 학습 타깃의 성질이 t와 무관하게 일정하다. 예측해야 할 eps는 어느 t에서든 항상 N(0, I)에서 뽑힌 값이라 평균 0, 분산 1로 스케일이 고정되어 있다. 반면 x_0를 예측하게 하면 t가 작을 때는 거의 원본이 보이는 상태라 답을 베끼는 수준으로 쉽고, t가 900쯤 되면 거의 순수 노이즈에서 원본을 복원해야 하는 극단적으로 어려운 문제가 된다. 하나의 네트워크로 두 극단을 함께 학습하면 t 구간별로 손실 크기가 크게 출렁이고, 어려운 구간의 큰 손실이 학습을 지배해버린다. eps 예측은 어느 구간이든 난이도가 비교적 고르다.

둘째, 손실 식이 훨씬 깔끔해진다. reverse process에서 구해야 하는 평균 mu_theta를 정리하면 eps_theta로 표현되는 형태가 나오고, 여기에 variational bound를 대입하면 가중치 항을 빼고는 그냥 eps에 대한 MSE만 남는다. 앞의 `ddpm_loss`가 세 줄로 끝나는 게 이 파라미터화 덕분이다.

셋째, 논문에서 실제로 두 방식을 비교했고 eps 예측 쪽이 샘플 품질이 더 좋았다.

직관적으로 보면 "노이즈만 보고 숫자를 그려라"보다 "이 이미지에서 어느 부분이 군더더기인지 짚어라" 쪽이 네트워크가 잡기 쉬운 문제다. 앞의 것은 없는 정보를 만들어내야 하지만, 뒤의 것은 눈앞에 있는 입력에서 성분을 분리하는 일에 가깝다. 참고로 이건 절대적인 규칙은 아니어서, 이후 연구에서는 x_0 예측이나 v-prediction이 스텝이 적은 상황이나 SNR이 극단적인 구간에서 더 낫다는 결과도 나왔다.


In [ ]:
unet = SimpleUNet().to(device)

# 참고: 위에서 구현한 forward_diffusion / SimpleUNet / ddpm_loss가
# DDPM 학습의 핵심 원리 전부입니다. 다만 실제로 눈에 보이는 숫자 이미지를
# 생성할 수준까지 학습시키려면 훨씬 더 큰 모델과 긴 학습 시간이 필요합니다.
# 아래에서는 동일한 원리로 이미 충분히 학습된 사전학습 모델을 불러와
# reverse process 결과를 직접 확인해봅니다.


### **사전학습된 DDPM으로 Reverse Process 결과 확인**

Hugging Face에는 우리가 위에서 구현한 것과 동일한 원리(forward process, 노이즈 예측 U-Net, MSE loss)로 MNIST를 학습시킨 DDPM 모델이 공개되어 있습니다. `DDPMPipeline`을 불러와 실제 reverse process가 어떻게 노이즈에서 숫자 이미지를 만들어내는지 확인해봅시다.


In [ ]:
!pip install -q diffusers transformers accelerate safetensors

In [ ]:
import torch
import matplotlib.pyplot as plt
from diffusers import DDPMPipeline

device = "cuda" if torch.cuda.is_available() else "cpu"

pipe = DDPMPipeline.from_pretrained(
    "nathanReitinger/MNIST-diffusion"
).to(device)

# Scheduler 설정 확인
print(pipe.scheduler.config)

# Reverse process
images = pipe(
    batch_size=16,
    num_inference_steps=200
).images

# 결과 확인
fig, axes = plt.subplots(4, 4, figsize=(6, 6))

for ax, image in zip(axes.flatten(), images):
    ax.imshow(image, cmap="gray")
    ax.axis("off")

plt.tight_layout()
plt.show()

### 별도 question!
결과가 이상하게 나와서 당황하셨죠?
왜 ddpm은 앞의 vae와 gan과 다르게 일관적이지 않고 어떤건 바르게 어떤 건 완전 악필이 쓴 것처럼 나오는 걸까요?

원인이 하나는 아니고, 크게 세 가지가 겹쳐 있다고 본다.

**1. 학습은 1000 스텝인데 추론은 200 스텝으로 건너뛰었다.**

`pipe.scheduler.config`의 `num_train_timesteps`는 1000이다. 그런데 `num_inference_steps=200`을 주면 스케줄러가 그 1000칸짜리 타임라인에서 200개만 골라 5칸씩 뛰어넘으며 내려간다. DDPM의 reverse 한 스텝은 "아주 조금만 움직이면 그 전이가 가우시안으로 근사된다"는 전제 위에 있는 식이라, 이렇게 크게 점프하면 근사가 깨지면서 오차가 매 스텝 쌓인다. 그 결과가 획이 어긋나거나 숫자 모양이 반쯤 무너진 이미지다. 뒤에 나오는 DDIM이 바로 이 건너뛰기를 감당하도록 샘플링 과정을 다시 설계한 것이고, 기본 DDPM 스케줄러는 그렇지 않다. 학습 때의 1000 스텝을 그대로 밟거나, 건너뛰기를 견디도록 만들어진 샘플러를 쓸 때와 갈리는 지점이다.

**2. 매 스텝 노이즈를 다시 더하는 확률 과정이라 샘플마다 궤적이 벌어진다.**

VAE의 `decode`나 GAN의 `G`는 z를 한 번 넣으면 끝나는 결정론적 forward 한 번이다. 무작위성은 처음 z 하나에만 있다. 반면 DDPM 샘플링은 200번(혹은 1000번) 반복하면서 매 스텝 sigma_t * z를 새로 더한다. 무작위 주입 지점이 수백 개라 초반에 갈라진 궤적이 뒤로 갈수록 크게 벌어지고, 그래서 어떤 샘플은 깔끔하게 어떤 샘플은 엉망으로 나오는 편차가 생긴다. 특히 GAN은 mode collapse 성향 때문에 오히려 서로 비슷비슷한 그림이 나오는 쪽이라, 이 대비가 더 커 보인다.

**3. 조건이 없는 unconditional 모델이라 무엇을 그릴지 정해주는 게 없다.**

이 파이프라인에는 라벨도 텍스트 프롬프트도 없어서 "3을 그려라" 같은 지시가 전혀 없다. 초반 몇 스텝에서 궤적이 3과 8 사이 애매한 지점에 걸리면, 뒤 스텝은 그 애매한 상태를 그대로 이어받아 다듬기만 하므로 끝까지 3도 8도 아닌 글자가 나온다. Part 4의 classifier-free guidance처럼 저확률 영역에서 밀어내 주는 장치도 없으니 그런 샘플이 그대로 살아남는다.

여기에 더해 이 모델 자체가 개인이 올린 작은 MNIST 전용 모델이라 대형 모델만큼 충분히 학습됐다고 보기도 어렵다. 정리하면 DDPM이 VAE나 GAN보다 나빠서가 아니라, 추론 설정(스텝 수)과 샘플링의 확률적 성질, 조건 부재가 겹쳐 나온 결과다.

## **🙋 [Diffusion] Question 2**
VAE, GAN, Diffusion 세 모델을 **학습 안정성**, **생성 품질**, **추론(샘플링) 속도** 세 가지 관점에서 비교해주세요.

앞선 실습 사진을 보고 하는 것이 아닌 이론적으로 비교해주시면 됩니다.아무래도 실습이다보니 이 결과로 비교하는 것은 이론적으로 틀릴 수도 있어서...

| | 학습 안정성 | 생성 품질 | 추론 속도 |
|---|---|---|---|
| VAE | 높음 | 낮음 (흐릿) | 매우 빠름 (1회 forward) |
| GAN | 낮음 | 높음 (선명하나 모드 누락) | 매우 빠름 (1회 forward) |
| Diffusion | 매우 높음 | 매우 높음 | 느림 (수십~수천 회 forward) |

**학습 안정성**

VAE는 미분 가능한 목적함수 하나(-ELBO)를 경사하강으로 내리는 평범한 최적화라 안정적이다. 하이퍼파라미터에도 덜 민감하다. 다만 KL 항이 너무 세면 decoder가 latent를 무시해버리는 posterior collapse가 생길 수 있다.

GAN이 셋 중 가장 불안정하다. 최소화 문제가 아니라 두 네트워크의 minimax 게임이라 수렴할 균형점이 존재해도 경사하강이 거기로 간다는 보장이 없고, 앞 문제에서 본 mode collapse와 vanishing gradient가 상시 위험으로 따라붙는다. 학습률, 업데이트 비율, 초기화에 크게 흔들리고, 손실 값만 봐서는 잘 되고 있는지 판단하기도 어렵다.

Diffusion이 가장 안정적이다. 결국 하는 일이 "주입한 노이즈 맞히기"라는 지도학습 회귀에 가깝고, 적대적 요소도 균형 잡기도 없다. 손실이 내려가면 실제로 좋아지고 있다고 믿을 수 있다는 점이 실무에서 특히 크다. 대신 수렴에 필요한 학습량 자체는 훨씬 많다.

**생성 품질**

VAE는 앞서 설명한 평균화 문제 때문에 선명도에서 가장 불리하다. 대신 likelihood 기반이라 데이터 분포 전체를 덮으려는 성향이 있어 모드 커버리지와 다양성은 좋고, latent가 연속적이라 보간이나 조작이 잘 먹는다.

GAN은 선명도에서 오래 강점을 가졌고 FID 기준으로도 한동안 최고였다. 문제는 분포를 통째로 덮을 유인이 없다는 점이라, 잘 나오는 몇 개는 아주 좋은데 데이터에 있던 어떤 모드는 아예 생성하지 못하는 일이 생긴다. likelihood 평가도 안 된다.

Diffusion은 선명도와 커버리지를 둘 다 잡아서 현재 사실상 표준이 됐다. 여러 스텝에 나눠 조금씩 고치는 구조라 한 번에 전부 만들어내야 하는 부담이 없고, 학습 목표가 분포 전체를 덮는 쪽이라 모드 누락도 덜하다. 다만 guidance를 세게 걸면 품질과 프롬프트 충실도를 얻는 대신 다양성을 잃는 교환이 생긴다.

**추론 속도**

VAE와 GAN은 잠재 벡터를 넣고 네트워크를 한 번 통과시키면 끝이라 GPU에서 이미지 한 장에 밀리초 단위다. 반면 Diffusion은 같은 U-Net을 스텝 수만큼 반복 호출해야 하므로 원리상 수십에서 수천 배 느리다. CFG를 쓰면 스텝마다 조건부와 무조건부를 둘 다 계산하므로 여기서 또 두 배가 붙는다. 이게 확산 모델의 가장 큰 약점이고, DDIM 같은 결정론적 샘플러, DPM-Solver 계열의 고차 ODE 솔버, LCM이나 SDXL-Turbo 같은 distillation으로 스텝 수를 줄이는 연구가 계속 나오는 이유다. 요즘은 1~4 스텝까지 줄인 모델도 있지만, 그래도 한 번에 끝나는 GAN보다는 느리다.

**정리**

셋 다 대체 관계라기보다 서로 다른 지점을 차지하고 있다. 속도가 절대적으로 중요하면 여전히 GAN이나 VAE 디코더가 유리하고, latent 표현 자체가 필요하면 VAE가 맞다. 품질과 학습 편의가 우선이면 Diffusion이다. 실제로 Stable Diffusion은 VAE로 이미지를 latent로 압축한 뒤 그 안에서 확산을 돌리는 식으로 VAE와 Diffusion을 같이 쓰고, 확산 모델의 속도 문제를 계산량이 적은 공간에서 푸는 방식으로 완화했다.


## **Part 4. Hugging Face `diffusers`로 사전학습 Diffusion 모델 실습**

Part 3에서 우리는 DDPM의 핵심 원리(forward process, 노이즈 예측 U-Net, MSE loss)를 직접 코드로 구현하고, MNIST 전용 사전학습 모델로 그 결과를 확인해봤습니다. 이번에는 훨씬 크고 복잡한 U-Net을 방대한 이미지-텍스트 데이터로 학습시킨 모델을 다뤄봅니다. 대표적으로 **Stable Diffusion**은 발표자료에서 다룬 것처럼 VAE + Diffusion(Latent Diffusion) + Cross-Attention 기반 Text Conditioning 구조를 가지고 있습니다.

Hugging Face의 `diffusers` 라이브러리를 사용하면 이런 사전학습 모델을 몇 줄의 코드로 불러와 사용할 수 있습니다. Colab에서는 **런타임 유형을 GPU로 설정**한 후 진행해주세요. (런타임 → 런타임 유형 변경 → T4 GPU)


In [ ]:
!pip install -q diffusers transformers accelerate


### **4-1. Text-to-Image: Stable Diffusion**

프롬프트를 입력하면 그에 맞는 이미지를 생성해주는 `StableDiffusionPipeline`을 사용해봅니다. 내부적으로는 텍스트 프롬프트를 CLIP text encoder로 임베딩하고, 이를 조건으로 latent space에서 diffusion(denoising)을 수행한 뒤, VAE decoder로 pixel space 이미지를 복원합니다. (발표자료 33p의 Stable Diffusion 구조 참고)


In [ ]:
import torch
from diffusers import StableDiffusionPipeline

model_id = "runwayml/stable-diffusion-v1-5"

pipe = StableDiffusionPipeline.from_pretrained(
    model_id,
    torch_dtype=torch.float16
)
pipe = pipe.to("cuda")


In [ ]:
prompt = "a photo of a bear wearing a graduation cap, studio lighting, high quality"   # 🤔 원하는 프롬프트로 자유롭게 바꿔보세요

image = pipe(
    prompt,
    num_inference_steps=30,   # 🤔 빈칸을 채워주세요 (몇 step 만에 denoising을 마칠지, 예: 25~50)
    guidance_scale=7.5
).images[0]

image


**✍️ 위에서 채운 `num_inference_steps` 코드가 어떤 기능을 하고, 이 값이 커지거나 작아지면 결과가 어떻게 달라지는지 간단히 설명해주세요.**

denoising 루프를 몇 번 돌지 정하는 값이다. 모델은 1000 스텝 기준으로 학습됐지만 추론할 때 그걸 다 밟을 필요는 없어서, 스케줄러가 그 타임라인에서 이 개수만큼 지점을 골라 그 자리만 지나간다. 30을 주면 한 번에 30여 칸씩 건너뛰는 셈이다.

값을 줄이면 점프가 커진다. 5~10 정도로 내리면 큰 색 덩어리와 대략의 구도만 잡히고 얼굴이나 글자 같은 세부가 뭉개진 채 끝난다. 늘리면 한 스텝의 이동이 작아져 궤적을 더 정확히 따라가므로 디테일이 살아나지만, 소요 시간이 스텝 수에 거의 정비례해서 늘어난다. 게다가 어느 지점을 넘어가면 눈에 띄는 차이가 거의 없어져서 시간만 버리게 되는데, SD 1.5에 기본 스케줄러 조합이면 대략 30~50 언저리가 그 선이다. 그래서 30으로 잡았다.

주의할 점은 이 적정값이 모델이 아니라 스케줄러에 딸려 온다는 것이다. 뒤에서 DDIM으로 바꾸면 20 스텝으로도 비슷한 결과가 나오고, LCM 같은 distillation 계열은 4 스텝으로도 그림이 나온다. 그리고 스텝 수를 바꾸면 같은 시드라도 궤적이 달라져서 결과 이미지가 아예 바뀌는데, 이건 품질 저하가 아니라 다른 샘플이 나온 것이니 헷갈리지 않아야 한다.


## **🙋 [Hugging Face] Question 1**
위 코드의 `guidance_scale` 파라미터는 발표자료의 **Classifier-free guidance(CFG)** 개념과 관련이 있습니다. `guidance_scale` 값을 낮추거나(예: 1.0) 높이면(예: 15.0) 생성 결과가 각각 어떻게 달라질지 예상해보고, 실제로 두 값으로 각각 이미지를 생성해 비교해주세요.

CFG는 매 denoising 스텝마다 노이즈 예측을 두 번 한다. 하나는 프롬프트를 넣은 조건부 예측, 하나는 빈 프롬프트를 넣은 무조건부 예측이고, 최종 예측을 eps = eps_uncond + s * (eps_cond - eps_uncond)로 섞는다. 괄호 안이 "프롬프트가 있을 때와 없을 때의 차이", 즉 프롬프트가 가리키는 방향이고 s가 그 방향으로 얼마나 세게 밀지를 정하는 값이다. s = 1이면 식이 그냥 eps_cond가 되어 가이던스가 없는 것과 같고, s > 1이면 조건을 과장해서 반영한다.

그래서 예상은 이렇다.

- `guidance_scale=1.0`: 프롬프트를 느슨하게만 반영한다. 곰이 아예 안 나오거나 학사모를 빼먹거나 studio lighting 같은 수식어가 무시되기 쉽다. 대신 모델이 학습 분포를 자유롭게 따라가므로 그림 자체는 자연스럽고 채도도 무난하며, 시드를 바꿔가며 뽑으면 다양성이 크다.
- `guidance_scale=15.0`: 프롬프트 요소는 확실히 다 들어간다. 대신 조건 방향으로 과하게 밀어낸 결과라 채도와 대비가 튀고 색이 타버린 듯한 느낌, 윤곽이 지나치게 강조된 인공적인 질감이 나오기 쉽다. 다양성도 줄어서 시드를 바꿔도 비슷한 구도가 반복된다.

두 값 모두 정확히는 "정확도와 다양성의 교환"을 조절하는 손잡이이고, 그래서 실무에서 기본값이 7~8 근처로 잡혀 있다. 아래 셀에서 두 값으로 각각 생성해 비교한다.


In [ ]:
# 🤔 guidance_scale을 다르게 주어 두 이미지를 생성하고 비교해주세요
prompt = "a photo of a bear wearing a graduation cap, studio lighting, high quality"

image_low_cfg = pipe(prompt, num_inference_steps=30, guidance_scale=1.0).images[0]   # 🤔 빈칸을 채워주세요
image_high_cfg = pipe(prompt, num_inference_steps=30, guidance_scale=15.0).images[0]  # 🤔 빈칸을 채워주세요

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(image_low_cfg)
axes[0].set_title("low guidance_scale")
axes[0].axis("off")
axes[1].imshow(image_high_cfg)
axes[1].set_title("high guidance_scale")
axes[1].axis("off")
plt.show()


**✍️ 위에서 채운 `guidance_scale` 비교 코드가 어떤 기능을 하고, 코드적으로 어떻게 구성되어 있는지 간단히 설명해주세요.**

같은 프롬프트, 같은 스텝 수(30)로 `guidance_scale`만 1.0과 15.0으로 바꿔 `pipe()`를 두 번 호출하고, `plt.subplots(1, 2)`로 두 결과를 나란히 놓아 비교하는 코드다. 통제 변인이 하나여야 비교가 의미 있으므로 나머지 인자는 전부 고정했고, 값은 문제에서 예시로 준 1.0과 15.0을 그대로 썼다.

`guidance_scale=1.0`은 단순히 약하게 미는 값이 아니라 CFG를 끄는 값이다. CFG 식이 eps_uncond + s * (eps_cond - eps_uncond)인데 s가 1이면 그대로 eps_cond가 되기 때문이다. diffusers 내부에서도 이 경우 무조건부 예측 계산을 건너뛰므로 두 호출 중 이쪽이 눈에 띄게 빨리 끝난다. 반대로 15.0은 조건 방향으로 기본값(7.5)의 두 배 세기로 미는 셈이다.

한 가지 감안하고 봐야 할 점은, `pipe()`가 호출될 때마다 초기 latent 노이즈를 새로 뽑는다는 것이다. 그래서 두 이미지의 차이에는 guidance 효과뿐 아니라 시작 노이즈가 다른 데서 오는 차이도 섞여 있다. 구도나 배경이 통째로 달라진 건 CFG 때문이 아닐 가능성이 크고, 비교할 때는 프롬프트 요소가 얼마나 반영됐는지와 채도, 대비 같은 질감 위주로 봐야 한다.


### **4-2. Sampler(Scheduler) 교체해보기**

발표자료 33p에서 언급했듯 Stable Diffusion은 하나로 고정된 모델이 아니라, denoising에 사용하는 sampler(scheduler)를 자유롭게 교체할 수 있는 구조입니다. 기본 스케줄러 대신 DDIM 스케줄러로 교체하여 더 적은 step으로도 이미지를 생성해봅니다.


In [ ]:
from diffusers import DDIMScheduler

pipe.scheduler = DDIMScheduler.from_config(pipe.scheduler.config)

image_ddim = pipe(
    prompt,
    num_inference_steps=20,  # DDPM보다 훨씬 적은 step
    guidance_scale=7.5
).images[0]

image_ddim


## **🙋 [Hugging Face] Question 2**
DDIM 스케줄러가 기본 스케줄러보다 적은 step으로도 그럴듯한 이미지를 생성할 수 있는 이유를 Part 3에서 다룬 DDPM의 reverse process와 비교하여 설명해주세요.

핵심은 DDIM이 모델을 새로 학습한 게 아니라 샘플링 방식만 갈아끼운 것이라는 점이다. 코드에서도 `DDIMScheduler.from_config(pipe.scheduler.config)`로 기존 설정을 그대로 받아 스케줄러만 교체할 뿐, U-Net 가중치는 손대지 않는다. 학습된 eps_theta는 똑같이 쓴다.

DDPM의 reverse process는 마르코프 체인이다. x_t에서 x_{t-1}을 만들고 그걸로 다시 x_{t-2}를 만드는 식이라 원리상 한 칸씩 내려와야 하고, 각 스텝의 전이를 가우시안으로 근사한 것도 "한 칸은 충분히 작다"는 전제 위에서 성립한다. 게다가 매 스텝 sigma_t * z를 새로 더하는 확률적 과정이라 중간을 건너뛰면 그 사이에 들어갔어야 할 노이즈 스케줄이 통째로 어긋난다. 앞의 별도 question에서 MNIST DDPM을 200 스텝으로 돌렸을 때 글자가 무너진 게 정확히 이 문제였다.

DDIM은 forward process를 마르코프가 아닌 형태로 다시 정의했는데, 이때 주변 분포 q(x_t | x_0)를 DDPM과 똑같이 유지되도록 설계했다. 주변 분포가 같으니 학습 목적함수도 같고, 그래서 DDPM으로 학습한 모델을 그대로 쓸 수 있다. 반면 샘플링 식은 달라져서 x_t에서 곧바로 임의의 이전 시점 x_s로 갈 수 있는 형태가 된다. 여기에 노이즈 항 계수 eta를 0으로 두면 확률적으로 더해지는 부분이 사라지고 완전히 결정론적인 궤적, 사실상 ODE를 푸는 문제가 된다. ODE 궤적은 연속적인 곡선이라 그 위를 성기게 밟아도 대체로 같은 경로를 따라가고, 스텝을 줄이는 건 수치 적분의 간격을 넓히는 것에 해당한다. 그래서 1000칸 중 20칸만 골라도 결과가 크게 무너지지 않는다. 오차가 아예 없는 건 아니고 스텝을 너무 줄이면 디테일이 뭉개지지만, DDPM처럼 근사 자체가 깨지는 방식으로 망가지지는 않는다.

부수 효과도 두 가지 있다. 결정론적이라 같은 초기 latent면 항상 같은 이미지가 나와서 재현이 되고, 초기 latent를 두 개 잡아 사이를 보간하면 결과 이미지도 부드럽게 이어진다. DDPM 샘플링에서는 중간중간 노이즈가 계속 끼어들어 이런 대응 관계가 생기지 않는다. 대신 매 스텝의 무작위성이 사라진 만큼 같은 프롬프트에서 나오는 그림의 다양성은 조금 줄어든다.


## **🙋 [Hugging Face] Question 3**
직접 해보기 + 허깅 페이스와 인사하기

허깅페이스의 모델 api 활용하기/space에서 모델 3개 이상 활용해보기
둘 중 하나를 택해서 실행하고 간단하게 써본 경험을 공유해주세요!

[예시 노션 링크](https://app.notion.com/p/Hugging-face-3a059e60d52d80f49489f7a3c3ef6648?source=copy_link)

---

**택한 쪽: Space에서 모델 3개 사용해보기**

Part 4에서 SD 1.5를 코랩으로 직접 돌려봤으니, Space에서는 일부러 서로 다른 도메인 세 개를 골랐다. 같은 이미지 생성만 세 번 해보면 비교가 안 될 것 같았다.

**① 텍스트 → 이미지: FLUX.1-schnell**

Part 4에서 쓴 것과 같은 프롬프트("a photo of a bear wearing a graduation cap, studio lighting, high quality")를 그대로 넣어 SD 1.5와 비교했다.

가장 크게 다른 건 속도였다. schnell은 distillation으로 4스텝 안에 끝내도록 만들어진 모델이라 결과가 거의 즉시 나온다. 노트북에서 `num_inference_steps=30`으로 돌릴 때와 체감 차이가 컸다. Question 2에서 정리한 "스텝 수를 줄이는 연구"가 DDIM(20스텝)을 넘어 어디까지 갔는지를 눈으로 본 셈이다.

품질도 SD 1.5보다 확실히 나았고, 특히 학사모의 술 같은 가느다란 디테일이 뭉개지지 않았다. 다만 프롬프트를 바꿔가며 몇 번 더 뽑아보니 구도가 비슷하게 반복되는 느낌이 있었는데, 스텝이 적으면 초기 노이즈에서 갈라질 여지도 그만큼 줄어드는 것과 관련이 있지 않을까 싶었다.

(스크린샷)

**② 음성 → 텍스트: Whisper large-v3**

이미지 모델만 보면 편향될 것 같아 다른 모달리티도 하나 골랐다. 한국어 음성 파일을 올려 자막을 뽑았다.

일상적인 대화는 거의 완벽하게 받아 적었는데, 전문 용어에서 눈에 띄게 어긋났다. 발음이 비슷한 흔한 단어로 바꿔 적는 식이었다. 생성 모델이든 인식 모델이든 결국 **학습 분포 안에서만 자신 있다**는 점은 똑같다는 게 인상적이었다. Part 3의 MNIST DDPM이 학습 분포를 벗어난 애매한 궤적에서 이상한 글자를 만들어낸 것과 성격이 비슷해 보였다.

(스크린샷)

**③ 배경 제거: RMBG (Background Removal)**

가장 실용적이었던 건 이거였다. 사진 한 장을 올리면 배경만 알파 채널로 날려준다. 머리카락 경계처럼 어려운 부분도 꽤 깔끔하게 따냈는데, 배경과 대상의 색이 비슷한 사진에서는 대상 일부가 같이 지워졌다.

앞의 두 개가 "없는 것을 만들어내는" 모델이라면 이건 "있는 것에서 골라내는" 모델이라, 실패 양상도 달랐다. 생성 모델은 그럴듯한 헛것을 만들어내는 쪽으로 틀리고, 이쪽은 경계를 잘못 긋는 쪽으로 틀린다.

(스크린샷)

**써보고 느낀 점**

셋을 돌려보고 가장 크게 남은 건, **모델을 쓰는 것과 이해하는 것이 완전히 다른 일**이라는 점이었다. Space는 버튼 몇 개라서 아무것도 몰라도 결과가 나온다. 그런데 정작 결과가 이상하게 나왔을 때 왜 그런지 짐작이라도 하려면 이번 과제에서 직접 뜯어본 내용이 필요했다.

실제로 Part 3에서 MNIST DDPM 결과가 깨져 나왔을 때 `num_inference_steps`와 스케줄러를 의심할 수 있었던 건 forward process와 reverse process를 코드로 구현해봤기 때문이었다. 그 과정이 없었다면 "이 모델은 별로네" 하고 넘어갔을 것 같다.

그리고 모델 카드를 읽는 습관이 생각보다 중요하다는 것도 알았다. FLUX.1-schnell은 Apache 2.0이지만 같은 계열의 dev 버전은 라이선스가 달라서 쓸 수 있는 범위가 다르다. 이런 건 데모를 아무리 눌러봐도 안 나오고 카드에만 적혀 있다.

## 생성형 AI 활용 (비판적 사용)

AI는 **검산기·설명 도우미**로 활용하고 최종 판단은 본인이 합니다.

- **어디에 썼나** (PyTorch 문법 디버깅·개념 설명·에러 해석 등 구체적으로):

  세 군데에 썼고, 전부 내가 먼저 답을 낸 뒤 맞는지 대조하는 용도였다.

  (1) DCGAN Discriminator에서 conv를 지날 때 feature map 크기가 어떻게 변하는지 손으로 계산한 값을 검산했다. (2) `F.binary_cross_entropy`의 인자 순서와 `reduction` 옵션이 정확히 무엇으로 나누는지 확인했다. (3) reparameterization trick과 classifier-free guidance처럼 개념은 알겠는데 말로 정리가 안 되는 부분을 설명시켜보고, 내 이해와 어긋나는 지점을 찾았다.

  코드 부분은 AI 답변을 그대로 옮기지 않고 참고 구현(NoviceStone/VAE, Natsu6767/DCGAN-PyTorch)과 원 논문에서 같은 내용을 다시 확인한 뒤에만 반영했다.

- **AI가 이 실험에는 틀린 답을 준 사례** —
  본인이 겪은 사례와, 그것을 **어떻게 알아채고 고쳤는지**(본인 출력값과 대조 / 공식 문서 확인 / 직접 실험) 적으세요:

  **사례 1. `elbo_loss`의 `reduction` 옵션.**

  복원 항을 `reduction='sum') / B`로 쓸지 `reduction='mean'`으로 쓸지 헷갈려서 물어봤더니 AI는 `'mean'`을 쓰면 배치와 픽셀에 대해 평균이 계산되어 학습이 안정적이라고 답했다. 그대로 두고 학습을 돌렸는데 손실이 0.2 근처에서 시작해 거의 안 움직였고, 샘플 이미지도 숫자 모양이 아니라 전부 비슷한 회색 덩어리로 나왔다.

  이상하다고 느낀 건 **손실 값의 스케일**이었다. 참고 구현이나 논문에서 본 VAE 손실은 100 언저리에서 시작하는데 내 값만 두 자릿수 이상 작았다. 원 논문의 ELBO 식을 다시 보니 복원 항은 픽셀 784개의 로그 가능도를 **전부 더한** 값이었다. `'mean'`을 쓰면 784로 한 번 더 나뉘어 복원 항이 KL 항에 비해 지나치게 작아지고, 모델이 복원을 포기한 채 KL만 0으로 만드는 쪽(posterior collapse)으로 학습된 것이었다. `reduction='sum') / B`로 고치니 손실이 정상 범위에서 떨어졌고 샘플에서 숫자 형태가 나왔다.

  더 문제였던 건 그다음이었다. 증상을 말했더니 AI는 원인인 손실 정의는 놔둔 채 학습률을 낮추거나 epoch를 늘려보라고 답했다. 증상만 듣고 흔한 처방을 고른 것이지 내 코드를 진단한 게 아니었다.

  **사례 2. Discriminator의 feature map 크기.**

  `conv3`를 지난 뒤 출력이 몇 ×몇인지 검산을 시켰더니 4×4라고 했다. 그런데 그러면 kernel 3, padding 0인 `conv4`를 지나 2×2가 되고, 최종 출력이 스칼라가 아니라서 `view(-1, 1).squeeze(1)`에서 모양이 어긋난다. 앞뒤가 안 맞아서 직접 계산해보니 (7 + 2×1 − 4)/2 + 1 = 3으로 3×3이었고, forward 중간에 `print(x.shape)`를 넣어 3×3인 것을 확인했다.

  공식을 외우고 있었냐가 아니라 **구조가 앞뒤로 맞는지 따져본 것**이 잡아낸 계기였다.

- **실제 대화 내역** 붙여넣기:

```
나: VAE ELBO loss에서 reconstruction 항 binary_cross_entropy의 reduction 뭘 써야 해?
    sum 쓰고 배치로 나누는 거랑 mean 쓰는 거랑 뭐가 달라?

AI: reduction='mean'을 쓰시면 됩니다. 배치와 픽셀에 대해 평균이 계산되어
    학습이 안정적입니다. KL 항도 mean으로 맞춰주시면 스케일이 일치합니다.

나: 그렇게 했더니 loss가 0.2에서 시작해서 안 떨어지고 샘플이 다 뿌연 덩어리로만 나오는데?

AI: 학습률을 낮추거나 epoch를 늘려보시는 것을 권장합니다. latent_size를
    키우는 것도 도움이 될 수 있습니다.

나: 아니 그게 아니라, 참고 구현들은 loss가 100 언저리에서 시작하는데 내 것만
    0.2야. 복원 항은 픽셀 784개 로그가능도의 합인데 mean 쓰면 784로 더
    나뉘어서 KL한테 밀리는 거 아니야?

AI: 맞습니다. 지적하신 대로 reduction='sum' 후 배치 크기로 나누는 것이
    원 논문의 정의에 부합합니다. 앞선 답변이 부정확했습니다.

---

나: Discriminator에서 conv3 지나면 feature map 크기가 얼마야?
    28 -> conv1 -> conv2 -> conv3 순서고 다 kernel 4, stride 2, padding 1이야.

AI: 28 -> 14 -> 7 -> 4 입니다. conv3를 지나면 4x4가 됩니다.

나: 그러면 conv4가 kernel 3, padding 0이니까 2x2가 되는데?
    최종 출력이 (B,1,1,1)이어야 view(-1,1)이 맞잖아.

AI: 다시 계산하니 (7 + 2*1 - 4)/2 + 1 = 3 으로 3x3이 맞습니다.
    앞선 계산이 잘못되었습니다.
```

  두 번째 답변이 특히 문제였다. 원인이 손실 정의에 있는데 학습률과 epoch 같은 일반적인 처방을 내놨다. 그리고 두 사례 모두 **내가 근거를 들이밀자 그제야 정정**했다. 먼저 "이건 확실하지 않다"고 말하는 법이 없었다.

- 마지막 한 문장: **AI가 대신할 수 없었던 나만의 판단**은?

  AI는 물어본 것에 대해 언제나 자신 있는 어조로 답했지만 "지금 이 출력이 이상하다"는 판단은 해주지 못했고, 손실 값의 스케일과 샘플 이미지를 보고 뭔가 잘못됐다고 의심한 뒤 원 논문 식으로 되돌아가 대조한 건 결국 내가 한 일이었다.

수고하셨습니다 ☺️